# ComplexTorch model-based validation

This notebook compares analytical measures computed from a known generating model with the same measures computed from a model inferred from simulated observations. Set `INFERENCE_MODEL` to `"VAR"` or `"SSM"`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import complextorch as ct
from complextorch.measures import integrate_spectral_mvgc

torch.set_default_dtype(torch.float64)
SEED=42
INFERENCE_MODEL="VAR"
N_VARIABLES=3
TRUE_ORDER=2
N_TIMES=6000
FREQUENCIES=torch.linspace(0.0,0.5,1025)
print(ct.__version__, INFERENCE_MODEL)


## 1. Ground-truth model and simulation


In [ ]:
A,Q=ct.demo_var(n_variables=N_VARIABLES,order=TRUE_ORDER)
true_model=ct.build_var_system(A,Q)
X=ct.simulate_var(A,Q,n_times=N_TIMES,burnin=1000,seed=SEED)
print("data",tuple(X.shape),"rho",true_model.spectral_radius.numpy())
fig,ax=plt.subplots(figsize=(11,4))
for i in range(N_VARIABLES): ax.plot(X[0,:500,i].numpy(),label=f"X{i}",lw=.8)
ax.legend(); ax.set_title("Simulated observations"); plt.show()


## 2. Analytical measures from the true model

All delays are configured independently, while the context computes one shared autocovariance sequence.


In [ ]:
config=ct.ModelMeasureConfig(
    frequencies=FREQUENCIES, autocovariance_max_lag=20,
    ais_lag=3, phiid_variables=(0,1), phiid_lag=2,
    cmem_max_lag=12, cmem_decomposition_max_lag=TRUE_ORDER,
    source=(1,), target=(0,), conditional=(2,),
    macro_projection=torch.tensor([[1.,1.,0.],[0.,.5,.5]])
)
true_context=ct.build_measure_context(true_model,config)
true_measures=ct.compute_all_model_measures(true_model,config,context=true_context)
print("max lag",true_context.max_lag)
print("available",true_measures["available"])
print("not available",true_measures["not_available"])


## 3. Infer a VAR or SSM from observations


In [ ]:
if INFERENCE_MODEL=="VAR":
    estimator=ct.VAR(order=TRUE_ORDER,solver="lstsq",covariance="mle",dtype="float64").fit(X[0])
    estimated_model=estimator.to_var_system()
else:
    n4sid=ct.N4SID(n_states=N_VARIABLES*TRUE_ORDER,block_rows=20).fit(X[0].cpu())
    em=ct.LinearGaussianEM(n4sid.system_,n_iter=20).fit(X[0].cpu())
    estimated_model=em.system_
estimated_context=ct.build_measure_context(estimated_model,config)
estimated_measures=ct.compute_all_model_measures(estimated_model,config,context=estimated_context)
print("available",estimated_measures["available"])


## 4. Compare invariant model-derived outputs


In [ ]:
def numeric(tree,prefix=""):
    out={}
    if isinstance(tree,torch.Tensor): out[prefix.rstrip(".")]=tree.detach().cpu()
    elif isinstance(tree,dict):
        for k,v in tree.items():
            if k not in {"context","available","not_available","model_type"}: out.update(numeric(v,prefix+k+"."))
    return out
T=numeric(true_measures); E=numeric(estimated_measures)
rows=[]
for key in sorted(set(T)&set(E)):
    if T[key].shape!=E[key].shape: continue
    a,b=T[key].double(),E[key].double()
    rmse=torch.sqrt(torch.mean((a-b)**2))
    scale=torch.sqrt(torch.mean(a**2)).clamp_min(1e-12)
    rows.append((key,str(tuple(a.shape)),float(rmse/scale)))
comparison=pd.DataFrame(rows,columns=["measure","shape","normalized_rmse"]).sort_values("normalized_rmse")
comparison


## 5. Visual recovery and internal consistency


In [ ]:
G0=true_measures["autocovariances"][0].cpu(); G1=estimated_measures["autocovariances"][0].cpu()
fig,ax=plt.subplots(figsize=(9,4))
for i in range(N_VARIABLES):
    ax.plot(G0[:,i,i],label=f"true X{i}")
    ax.plot(G1[:,i,i],"--",label=f"estimated X{i}")
ax.legend(ncol=2); ax.set_title("Autocovariance recovery"); plt.show()
for label,measures in [("true",true_measures),("estimated",estimated_measures)]:
    if "mvgc" in measures and "spectral" in measures["mvgc"]:
        integrated=integrate_spectral_mvgc(measures["mvgc"]["spectral"],FREQUENCIES)
        torch.testing.assert_close(integrated,measures["mvgc"]["temporal"],rtol=1e-5,atol=1e-7)
        print(label,"spectral MVGC integrates to temporal MVGC")


## Interpretation

The comparison is performed on observable covariances, autocovariances, spectra and derived measures rather than directly on latent matrices. Secondary observation-refit estimators should remain separate from the primary model-based API.
